In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from solver import *

# PDE in Neumann conditions

### 1D Stationary Laplace equation

In [ ]:
# Symbols
x, xi = symbols('x xi', real=True)
u = Function('u')(x)

# Small regularisation parameter (same as in 2D example)
epsilon = 0.0001

# Symbol of the operator: -d²/dx² + ε  ↔  ξ² + ε
# Right‑hand side
f = (pi**2 + epsilon) * cos(pi * x)

# PDE: (ξ² + ε) u = f
eq = Eq(psiOp(xi**2 + epsilon, u), f)

solver = PDESolver(eq)

Lx = 2.0        # domain [-1, 1]
Nx = 64

solver.setup(
    Lx=Lx, Nx=Nx,
    boundary_condition='neumann',
    initial_condition=None
)

# Solve stationary problem using asymptotic inversion
u_num = solver.solve_stationary_psiOp(order=3)

# Exact solution
def u_exact(x):
    return np.cos(np.pi * x)

# Test and visualise
solver.test(u_exact=u_exact, threshold=1e-2, component='real')
solver.show_stationary_solution(u=u_num, component='real')

## 1D Heat equation

In [ ]:
import numpy as np
from scipy.fftpack import dct, idct

# Import your patched PDESolver (either from your file or the class above)
from solver import PDESolver   # after applying the patch

# ----------------------------------------------------------------------
# 1. Define the PDE
# ----------------------------------------------------------------------
t, x, xi = symbols('t x xi', real=True)
u = Function('u')(t, x)

eq = Eq(diff(u, t), psiOp(xi**2 + 0.0001, u))

solver = PDESolver(eq)

# ----------------------------------------------------------------------
# 2. Parameters
# ----------------------------------------------------------------------
Lx = 2.0          # domain [-1, 1]
Nx = 256
Lt = 1.0
Nt = 100

# Initial condition: Gaussian centered at 0
def initial_condition(x):
    return np.exp(- (x / 0.3)**2)   # sigma = 0.3

# ----------------------------------------------------------------------
# 3. Setup the solver (Neumann BC)
# ----------------------------------------------------------------------
solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    boundary_condition='neumann',
    initial_condition=initial_condition
)

# ----------------------------------------------------------------------
# 4. Precompute the exact solution via cosine series
# ----------------------------------------------------------------------
# The exact solution for a given initial condition f(x) is:
#   u(x,t) = Σ_{k=0}^{N-1} a_k cos(kπ (x+L/2)/L) exp(-λ_k t)
# where a_k = DCT type II (orthonormal) of f.
#
# We'll compute the coefficients once and then evaluate the series at any (x,t).

# Grid used by the solver (same as self.x_grid)
x_grid = solver.x_grid

# Compute DCT coefficients (orthonormal, type II)
a_k = dct(initial_condition(x_grid), type=2, norm='ortho')

# Eigenvalues: λ_k = (kπ/L)^2 + ε
k_vals = np.arange(Nx) * np.pi / Lx
lambda_k = k_vals**2 + 0.0001

def u_exact(x, t):
    """
    Evaluate the exact solution at arbitrary x and t using the cosine series.
    x can be a scalar or a 1D array.
    """
    # Ensure x is an array for broadcasting
    x_arr = np.asarray(x)
    # Compute cosine basis for all k
    # cos(kπ (x+L/2)/L) = cos(kπ (x+1)/2) because L=2
    # but we keep the general formula for clarity
    cos_terms = np.cos(k_vals * (x_arr[:, None] + Lx/2) / (Lx/2))   # shape (len(x), Nx)
    # Sum over k
    u_vals = np.dot(cos_terms, a_k * np.exp(-lambda_k * t))
    return u_vals

# ----------------------------------------------------------------------
# 5. Run the solver
# ----------------------------------------------------------------------
solver.solve()

# ----------------------------------------------------------------------
# 6. Test against exact solution at several times
# ----------------------------------------------------------------------
n_test = 4
threshold = 100

for i in range(n_test + 1):
    t_eval = i * Lt / n_test
    # The solver stores frames; we can use the built-in test method,
    # but we need to provide an exact function that works on (x,t).
    # The test method expects u_exact(x[,y][,t]).
    # For 1D non‑stationary, we can pass a function of (x, t).
    error = solver.test(
        u_exact=lambda x, t_val: u_exact(x, t_val),
        t_eval=t_eval,
        norm='relative',
        threshold=threshold,
        component='real'
    )
    print(f"t = {t_eval:.2f}, relative error = {error:.2e}")

# ----------------------------------------------------------------------
# 7. Visualize the final solution
# ----------------------------------------------------------------------
plt.figure(figsize=(10,4))
plt.plot(solver.x_grid, solver.u_prev, label='Numerical')
plt.plot(solver.x_grid, u_exact(solver.x_grid, Lt), '--', label='Exact')
plt.xlabel('x')
plt.ylabel('u(x,t)')
plt.title(f'Solution at t = {Lt}')
plt.legend()
plt.grid()
plt.show()

## 1D Wave equation

In [ ]:
from sympy import symbols, Function, diff, Eq, pi
import numpy as np
from psiop import psiOp
from solver import PDESolver

t, x, xi = symbols('t x xi', real=True)
u = Function('u')(t, x)

# Wave equation: ∂²u/∂t² = ∂²u/∂x²   (c = 1)
# ∂²/∂x² = -psiOp(xi**2, u)
eq = Eq(diff(u, t, t), -psiOp(xi**2, u))

solver = PDESolver(eq)

Lx = np.pi
Nx = 128
Lt = 2.0
Nt = 200

def initial_condition(x):
    return np.cos(2*x)

def initial_velocity(x):
    return 0.0

def u_exact(x, t):
    return np.cos(2*x) * np.cos(2*t)

solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='neumann',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity
)

solver.solve()

# Plot energy evolution
solver.plot_energy() # (log=True)


# Automatic tests
n_test = 4
for i in range(n_test + 1):
    solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=0.1, component='real')

## 2D Stationary Laplace equation

In [ ]:
from sympy import symbols, Function, Eq, pi, cos
import numpy as np
from psiop import psiOp
from solver import PDESolver

# Symbols
x, y, xi, eta = symbols('x y xi eta', real=True)
u = Function('u')(x, y)

# Symbol of -Δ is xi² + eta²
f = 2*pi**2 * cos(pi*x) * cos(pi*y)
eq = Eq(psiOp(xi**2 + eta**2 + 0.0001, u), f)

solver = PDESolver(eq)

Lx = 2.0      # domain [-1,1] in x
Ly = 2.0      # domain [-1,1] in y
Nx = 64
Ny = 64

solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny,
    boundary_condition='neumann',
    initial_condition=None
)

# Solve stationary problem using asymptotic inversion
u_num = solver.solve_stationary_psiOp(order=3)

# Exact solution
def u_exact(x, y):
    return np.cos(np.pi*x) * np.cos(np.pi*y)

solver.test(u_exact=u_exact, threshold=1e-2, component='real')
solver.show_stationary_solution(u=u_num, component='real')

## 2D Heat equation

In [ ]:
from sympy import symbols, Function, diff, Eq, pi, cos
import numpy as np
from psiop import psiOp
from solver import PDESolver

t, x, y, xi, eta = symbols('t x y xi eta', real=True)
u = Function('u')(t, x, y)

eq = Eq(diff(u, t), -psiOp(xi**2 + eta**2 + 0.0001, u))

solver = PDESolver(eq)

Lx, Ly = 2.0, 2.0
Nx, Ny = 64, 64
Lt, Nt = 0.5, 50

def initial_condition(x, y):
    return np.cos(np.pi * x) * np.cos(2 * np.pi * y)   # wavenumbers π/Lx, 2π/Ly

def u_exact(x, y, t):
    kx, ky = np.pi / Lx, 2 * np.pi / Ly       # = π/2, π
    decay = (kx**2 + ky**2 + 0.0001) * t
    return np.cos(kx * x) * np.cos(ky * y) * np.exp(-decay)

solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='neumann',
    initial_condition=initial_condition
)

solver.solve()
# Automatic tests
n_test = 4
for i in range(n_test + 1):
    solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=10, component='real')

## 2D Wave equation

In [ ]:
from sympy import symbols, Function, diff, Eq, pi, cos
import numpy as np
from psiop import psiOp
from solver import PDESolver

t, x, y, xi, eta = symbols('t x y xi eta', real=True)
u = Function('u')(t, x, y)

# 2D wave equation: u_tt = u_xx + u_yy
eq = Eq(diff(u, t, t), -psiOp(xi**2 + eta**2, u))

solver = PDESolver(eq)

Lx, Ly = 2.0, 2.0
Nx, Ny = 64, 64
Lt, Nt = 1.0, 200   # final time = 1.0, 200 steps

def initial_condition(x, y):
    return np.cos(2*np.pi*x) * np.cos(3*np.pi*y)

def initial_velocity(x, y):
    return 0.0  # zero initial velocity

def u_exact(x, y, t):
    omega = np.sqrt(13) * np.pi
    return np.cos(2*np.pi*x) * np.cos(3*np.pi*y) * np.cos(omega * t)

solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='neumann',   # Fourier spectral method works best with periodic BC
    initial_condition=initial_condition,
    initial_velocity=initial_velocity
)

solver.solve()

# Plot energy evolution
solver.plot_energy() # (log=True)

# Test at a few times (the exact solution should be reproduced with high accuracy)
n_test = 4
for i in range(n_test + 1):
    solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=10, component='real')